In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import VarianceThreshold
import pickle
import warnings

warnings.filterwarnings('ignore')


In [2]:
df = pd.read_pickle('output/processed_data.pkl')

print(f"✓ Loaded {len(df):,} transactions")
print(f"✓ Starting features: {len(df.columns)}")

original_cols = len(df.columns)


✓ Loaded 590,540 transactions
✓ Starting features: 436


In [3]:
df['amt_log'] = np.log1p(df['TransactionAmt'])
df['amt_decimal'] = df['TransactionAmt'] - df['TransactionAmt'].astype(int)

df['is_round'] = (df['TransactionAmt'] % 1 == 0).astype(int)
df['is_round_10'] = (df['TransactionAmt'] % 10 == 0).astype(int)
df['is_round_100'] = (df['TransactionAmt'] % 100 == 0).astype(int)

df['amt_category'] = pd.cut(
    df['TransactionAmt'],
    bins=[0, 50, 100, 200, 500, 1000, np.inf],
    labels=['very_low', 'low', 'medium', 'high', 'very_high', 'extreme']
)

print("✓ Amount-based features created")


✓ Amount-based features created


In [4]:
df['hour'] = (df['TransactionDT'] % (24*3600)) / 3600
df['day'] = df['TransactionDT'] / (24*3600)
df['day_of_week'] = (df['day'] % 7).astype(int)

df['is_night'] = ((df['hour'] >= 0) & (df['hour'] < 6)).astype(int)
df['is_morning'] = ((df['hour'] >= 6) & (df['hour'] < 12)).astype(int)
df['is_afternoon'] = ((df['hour'] >= 12) & (df['hour'] < 18)).astype(int)
df['is_evening'] = ((df['hour'] >= 18) & (df['hour'] <= 24)).astype(int)

df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

df['is_business_hours'] = (
    (df['hour'] >= 9) & 
    (df['hour'] < 17) & 
    (df['day_of_week'] < 5)
).astype(int)

print("✓ Time-based features created")


✓ Time-based features created


In [5]:
df['card1_card2'] = df['card1'].astype(str) + '_' + df['card2'].astype(str)
df['card1_card3'] = df['card1'].astype(str) + '_' + df['card3'].astype(str)

card_counts = df['card1'].value_counts()
df['card1_count'] = df['card1'].map(card_counts)

print("✓ Card features created")


✓ Card features created


In [6]:
df['addr_diff'] = df['addr1'] - df['addr2']
df['addr_match'] = (df['addr1'] == df['addr2']).astype(int)
df['addr_combo'] = df['addr1'].astype(str) + '_' + df['addr2'].astype(str)

print("✓ Address features created")


✓ Address features created


In [7]:
df['email_match'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)

popular_providers = ['gmail.com', 'yahoo.com', 'hotmail.com', 'outlook.com']
df['email_is_popular'] = df['P_emaildomain'].isin(popular_providers).astype(int)

print("✓ Email features created")


✓ Email features created


In [8]:
df['dist_diff'] = df['dist1'] - df['dist2']
df['dist_ratio'] = df['dist1'] / (df['dist2'] + 1)

print("✓ Distance features created")


✓ Distance features created


In [10]:
card_stats = df.groupby('card1')['TransactionAmt'].agg(['mean', 'std', 'count'])
card_stats.columns = ['card1_amt_mean', 'card1_amt_std', 'card1_count_agg']
df = df.merge(card_stats, on='card1', how='left')

addr_stats = df.groupby('addr1')['TransactionAmt'].agg(['mean', 'count'])
addr_stats.columns = ['addr1_amt_mean', 'addr1_count']
df = df.merge(addr_stats, on='addr1', how='left')

prod_stats = df.groupby('ProductCD')['TransactionAmt'].agg(['mean', 'count'])
prod_stats.columns = ['product_amt_mean', 'product_count']
df = df.merge(prod_stats, on='ProductCD', how='left')

email_stats = df.groupby('P_emaildomain')['TransactionAmt'].agg(['mean', 'count'])
email_stats.columns = ['email_amt_mean', 'email_count']
df = df.merge(email_stats, on='P_emaildomain', how='left')

print("✓ Aggregation features created")


✓ Aggregation features created


In [11]:
"""d_cols = [c for c in df.columns if c.startswith('D') and c[1:].isdigit()]
df['d_null_count'] = df[d_cols].isnull().sum(axis=1)

v_cols = [c for c in df.columns if c.startswith('V') and c[1:].isdigit()]
df['v_null_count'] = df[v_cols].isnull().sum(axis=1)
df['v_mean'] = df[v_cols].mean(axis=1)
df['v_std'] = df[v_cols].std(axis=1)

m_cols = [c for c in df.columns if c.startswith('M') and c[1:].isdigit()]
for col in m_cols:
    df[col] = df[col].map({'T': 1, 'F': 0}).fillna(-1)

df['m_sum'] = df[m_cols].sum(axis=1)
df['m_null_count'] = (df[m_cols] == -1).sum(axis=1)

print("✓ D, V, M features processed")
"""



import numpy as np

# =========================
# D columns (SAFE)
# =========================
d_cols = [c for c in df.columns if c.startswith('D') and c[1:].isdigit()]

if len(d_cols) > 0:
    df['d_null_count'] = df[d_cols].isna().sum(axis=1).astype(np.int16)

print(f"✓ D columns processed: {len(d_cols)}")


# =========================
# V columns (MEMORY SAFE)
# =========================
v_cols = [c for c in df.columns if c.startswith('V') and c[1:].isdigit()]

if len(v_cols) > 0:
    # Convert to float32 to reduce memory
    df[v_cols] = df[v_cols].astype(np.float32)

    df['v_null_count'] = df[v_cols].isna().sum(axis=1).astype(np.int16)

    # ⚠️ Use numpy for faster + lower memory ops
    v_values = df[v_cols].values

    df['v_mean'] = np.nanmean(v_values, axis=1).astype(np.float32)
    df['v_std'] = np.nanstd(v_values, axis=1).astype(np.float32)

    del v_values  # 🔥 free memory immediately

print(f"✓ V columns processed: {len(v_cols)}")


# =========================
# M columns (SAFE)
# =========================
m_cols = [c for c in df.columns if c.startswith('M') and c[1:].isdigit()]

if len(m_cols) > 0:
    for col in m_cols:
        df[col] = df[col].map({'T': 1, 'F': 0}).fillna(-1).astype(np.int8)

    df['m_sum'] = df[m_cols].sum(axis=1).astype(np.int16)
    df['m_null_count'] = (df[m_cols] == -1).sum(axis=1).astype(np.int16)

print(f"✓ M columns processed: {len(m_cols)}")
print("✓ D, V, M features processed safely")


✓ D columns processed: 15
✓ V columns processed: 339
✓ M columns processed: 9
✓ D, V, M features processed safely


In [12]:
# Separate target
y = df['isFraud']
X = df.drop(['isFraud', 'TransactionID'], axis=1, errors='ignore')

# Identify column types
numeric_cols = X.select_dtypes(include=[np.number]).columns
categorical_cols = X.select_dtypes(include=['object', 'category']).columns

# Fill numeric missing values
X[numeric_cols] = X[numeric_cols].fillna(-999)

# 🔥 FIX: convert categorical → string before fillna
for col in categorical_cols:
    X[col] = X[col].astype(str).fillna('unknown')

print("✓ Missing values handled safely")


✓ Missing values handled safely


In [13]:
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

print("✓ Categorical encoding complete")


✓ Categorical encoding complete


In [14]:
selector = VarianceThreshold(0)
X_selected = selector.fit_transform(X)

selected_features = X.columns[selector.get_support()]
X = X[selected_features]

print(f"Features after selection: {len(X.columns)}")


Features after selection: 477


In [15]:
X.to_pickle('output/X_train.pkl')
y.to_pickle('output/y_train.pkl')

with open('output/label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)

with open('output/feature_names.pkl', 'wb') as f:
    pickle.dump(list(X.columns), f)

print("✓ All feature engineering outputs saved")


✓ All feature engineering outputs saved
